# MALT overview

A first look at the `default` config of [MALT](https://huggingface.co/datasets/metr-evals/malt-public): 7,179 agent runs across 169 tasks and ~19 models, labelled for behaviours such as reward hacking, sabotage and giving up.

**Structure.** Each row is one run with
- `metadata`: `run_id`, `task_id`, `model`, `labels`, `manually_reviewed`, `run_source` (`prompted` = the model was told to misbehave, `unprompted` = organic), `has_chain_of_thought`
- `samples`: a list of model calls. Each has `input` (the messages sent to the API) and `output` (a list of completions, each a list of messages).

**Scaffolds differ.** Some scaffolds resend the whole conversation each step, so the last sample's input is the full transcript. Others (e.g. tree-search agents) send independent single-message prompts. The length stats below therefore count *model-generated output* across all samples and do not rely on the last sample.

The full config is about 20 GB uncompressed. The notebook therefore loads `metadata` for every run (fast), but reads `samples` only from `SAMPLE_FILES`, the first parquet file by default.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.compute as pc
import pyarrow.parquet as pq

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 80)

DATA_DIR = Path("../data/malt-public/data")
FILES = sorted(DATA_DIR.glob("*.parquet"))
# Transcripts are ~20 GB uncompressed; anything reading `samples` uses only this subset
SAMPLE_FILES = FILES[:1]
print(f"{len(FILES)} parquet files; reading samples from {[f.name for f in SAMPLE_FILES]}")

## Run metadata

In [ ]:
frames = []
for f in FILES:
    m = pq.read_table(f, columns=["metadata"]).flatten().to_pandas()
    m.columns = [c.removeprefix("metadata.") for c in m.columns]
    m["file"] = f.name
    frames.append(m)
runs = pd.concat(frames, ignore_index=True)

runs["task_family"] = runs["task_id"].str.split("/").str[0]
runs["labels_str"] = runs["labels"].map(lambda ls: ", ".join(sorted(ls)))
runs["is_normal"] = runs["labels"].map(lambda ls: list(ls) == ["normal"])
runs.head()

In [ ]:
print(f"runs:         {len(runs):,}")
print(f"tasks:        {runs.task_id.nunique()}")
print(f"task families: {runs.task_family.nunique()}")
print(f"models:       {runs.model.nunique()}")
print()
for col in ["run_source", "manually_reviewed", "has_chain_of_thought", "is_normal"]:
    print(runs[col].value_counts().to_string(), end="\n\n")

### Labels

A run can have several labels. `normal` means no notable behaviour was found.

In [ ]:
labels = runs.explode("labels", ignore_index=True).rename(columns={"labels": "label"})
label_counts = labels.groupby(["label", "run_source"]).size().unstack(fill_value=0)
label_counts["total"] = label_counts.sum(axis=1)
label_counts.sort_values("total", ascending=False)

In [ ]:
ax = (label_counts.drop(columns="total").drop(index="normal", errors="ignore")
      .sort_values("prompted").plot.barh(stacked=True, figsize=(8, 4)))
ax.set(title="Non-normal labels (runs)", xlabel="runs", ylabel="")
plt.tight_layout()

In [ ]:
# Label co-occurrence among non-normal runs
runs.loc[~runs.is_normal, "labels_str"].value_counts()

In [ ]:
# Label counts per parquet file (file number parsed from e.g. "public-00007-of-00042.parquet")
from matplotlib.colors import LogNorm

labels["file_num"] = labels["file"].str.extract(r"public-(\d+)-of", expand=False).astype(int)
file_label = pd.crosstab(labels["label"], labels["file_num"])
file_label = file_label.loc[file_label.sum(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(16, 4.5))
im = ax.imshow(file_label.where(file_label > 0), cmap="Blues", norm=LogNorm(vmin=0.5, vmax=file_label.values.max()), aspect="auto")
for (i, j), n in np.ndenumerate(file_label.values):
    if n:
        ax.text(j, i, n, ha="center", va="center", fontsize=6.5,
                color="white" if n > np.sqrt(file_label.values.max()) else "#333")
ax.set_xticks(range(file_label.shape[1]), file_label.columns, fontsize=8)
ax.set_yticks(range(file_label.shape[0]), file_label.index)
ax.set_xticks(np.arange(-0.5, file_label.shape[1]), minor=True)
ax.set_yticks(np.arange(-0.5, file_label.shape[0]), minor=True)
ax.grid(which="minor", color="white", linewidth=1.5)
ax.tick_params(which="minor", length=0)
ax.spines[:].set_visible(False)
ax.set(xlabel="file number", title="Runs per label in each parquet file (log color scale)")
fig.colorbar(im, ax=ax, label="runs", pad=0.01)
plt.tight_layout()

### Models

In [ ]:
model_label = pd.crosstab(labels["model"], labels["label"], margins=True, margins_name="total")
model_label.sort_values("total", ascending=False)

In [ ]:
# Fraction of each model's runs that are non-normal, split by prompted/unprompted
(runs.groupby(["model", "run_source"])
     .agg(runs=("run_id", "size"), frac_flagged=("is_normal", lambda s: 1 - s.mean()),
          cot=("has_chain_of_thought", "mean"), reviewed=("manually_reviewed", "mean"))
     .round(2)
     .sort_values("runs", ascending=False))

### Tasks

In [ ]:
family = (runs.groupby("task_family")
              .agg(runs=("run_id", "size"), tasks=("task_id", "nunique"), models=("model", "nunique"),
                   frac_flagged=("is_normal", lambda s: 1 - s.mean()))
              .round(2)
              .sort_values("runs", ascending=False))
family

In [ ]:
# Where do the unprompted (organic) misbehaviours show up?
unprompted_flagged = labels[(labels.run_source == "unprompted") & (labels.label != "normal")]
pd.crosstab(unprompted_flagged["task_family"], unprompted_flagged["label"])

## Transcript lengths

Computed on `SAMPLE_FILES` only (171 runs from the first file by default), so treat the numbers as indicative. Models that aren't in the subset won't appear.

- `n_samples`: model calls in the run
- `n_completions`: completions returned (a call can return several)
- `n_tool_calls`: output messages carrying a `function_call`
- `output_chars`: total characters the model generated
- `max_input_chars`: largest prompt the model saw (approximates final context size for linear scaffolds)

In [ ]:
def transcript_stats(path):
    """Per-run length stats for one parquet file, computed in Arrow without materialising Python objects."""
    t = pq.read_table(path, columns=["samples", "metadata"])
    samples = t.column("samples").combine_chunks()
    n_runs = len(samples)
    flat = pc.list_flatten(samples)
    sample_run = pc.list_parent_indices(samples).to_numpy()

    # output: per sample, a list of completions, each a list of messages
    outputs = flat.field("output")
    completion_sample = pc.list_parent_indices(outputs).to_numpy()
    completions = pc.list_flatten(outputs)
    out_msgs = pc.list_flatten(completions)
    out_msg_run = sample_run[completion_sample[pc.list_parent_indices(completions).to_numpy()]]
    out_chars = pc.fill_null(pc.utf8_length(out_msgs.field("content")), 0).to_numpy(zero_copy_only=False)
    has_call = pc.is_valid(out_msgs.field("function_call").field("name")).to_numpy(zero_copy_only=False)

    # input: total characters in each sample's prompt
    in_msgs = flat.field("input")
    in_chars = np.bincount(
        pc.list_parent_indices(in_msgs).to_numpy(),
        weights=pc.fill_null(pc.utf8_length(pc.list_flatten(in_msgs).field("content")), 0).to_numpy(zero_copy_only=False),
        minlength=len(flat),
    )
    max_in_chars = pd.Series(in_chars).groupby(sample_run).max().reindex(range(n_runs), fill_value=0)

    per_run = lambda w: np.bincount(out_msg_run, weights=w, minlength=n_runs).astype(int)
    return pd.DataFrame({
        "run_id": t.column("metadata").combine_chunks().field("run_id").to_numpy(),
        "n_samples": pc.list_value_length(samples).to_numpy(),
        "n_completions": np.bincount(sample_run[completion_sample], minlength=n_runs),
        "n_tool_calls": per_run(has_call),
        "output_chars": per_run(out_chars),
        "max_input_chars": max_in_chars.to_numpy().astype(int),
    })


lengths = pd.concat([transcript_stats(f) for f in SAMPLE_FILES], ignore_index=True)
subset = runs.merge(lengths, on="run_id")
print(f"{len(subset)} runs in subset")
subset[lengths.columns.drop("run_id")].describe().round(0)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["n_samples", "output_chars", "max_input_chars"]):
    ax.hist(subset[col].clip(lower=1), bins=np.logspace(0, np.log10(subset[col].max()), 50))
    ax.set(xscale="log", title=col, xlabel=col, ylabel="runs")
plt.tight_layout()

In [ ]:
(subset.groupby("model")[["n_samples", "n_tool_calls", "output_chars", "max_input_chars"]]
     .median()
     .round()
     .sort_values("output_chars", ascending=False))

In [ ]:
# Are flagged runs longer or shorter than normal ones?
subset.groupby(["run_source", "is_normal"])[["n_samples", "output_chars"]].median().round()

## Reading a transcript

`show_sample` prints one sample (model call): its input messages and the completions that came back. For linear scaffolds, `sample=-1` shows the whole conversation. Use `pick_run` to sample a run with a given label from `SAMPLE_FILES`.

In [ ]:
def load_run(run_id):
    """Load one run's row (metadata + samples) as Python objects."""
    # Read only the row group holding the run; some row groups are ~1 GB uncompressed
    pf = pq.ParquetFile(DATA_DIR / runs.loc[runs.run_id == run_id, "file"].item())
    for i in range(pf.num_row_groups):
        ids = pf.read_row_group(i, columns=["metadata"]).column("metadata").combine_chunks().field("run_id")
        idx = pc.index(ids, run_id).as_py()
        if idx >= 0:
            return pf.read_row_group(i).slice(idx, 1).to_pylist()[0]
    raise KeyError(run_id)


def pick_run(label=None, model=None, run_source=None, seed=None):
    df = runs[runs.file.isin([f.name for f in SAMPLE_FILES])]
    if label is not None:
        df = df[df["labels"].map(lambda ls: label in ls)]
    if model is not None:
        df = df[df.model == model]
    if run_source is not None:
        df = df[df.run_source == run_source]
    return int(df.sample(1, random_state=seed).run_id.item())


def _fmt(msg, max_chars=None):
    text = msg["content"] or ""
    if max_chars is not None and len(text) > max_chars:
        text = text[:max_chars] + f"\n… [{len(text) - max_chars:,} more chars]"
    fc = msg.get("function_call")
    if fc and fc.get("name"):
        text += f"\n→ {fc['name']}({fc['arguments']})"
    tag = msg["role"] + (f" / {msg['name']}" if msg.get("name") else "")
    if (msg.get("metadata") or {}).get("completion_type") == "reasoning_completion":
        tag += " · reasoning"
    return f"[{tag}]\n{text}\n"


def show_sample(run, sample=-1, max_chars=1500):
    m = run["metadata"]
    print(f"run {m['run_id']} | {m['model']} | {m['task_id']} | labels={m['labels']} | {m['run_source']}")
    s = run["samples"][sample]
    print(f"sample {sample} of {len(run['samples'])}: {len(s['input'])} input messages, {len(s['output'])} completion(s)\n")
    print("=" * 30, "INPUT", "=" * 30)
    for msg in s["input"]:
        print(_fmt(msg, max_chars))
    for i, completion in enumerate(s["output"]):
        print("=" * 30, f"OUTPUT {i}", "=" * 30)
        for msg in completion:
            print(_fmt(msg, max_chars))

In [ ]:
# The first file only has normal and gives_up runs
run = load_run(pick_run(label="gives_up", seed=0))
show_sample(run, sample=-1)

### Example transcripts to a Markdown file

The next cell picks one run for each non-normal label below and writes its `show_sample` output (last call, each message cut at 1,500 characters) to `outputs/malt_examples.md`. Runs are drawn from all files. Unprompted runs are used whenever a label has any, and runs carrying only that label are preferred. `refusals` and `sabotage` exist only as prompted runs. Change `seed` to draw different runs.

In [ ]:
import re

EXAMPLE_LABELS = ["refusals", "reasoning_about_task", "sabotage", "ignores_task_instructions", "bypass_constraints", "hardcoded_solution"]
EXAMPLES_PATH = Path("../outputs/malt_examples.md")


def pick_example(label, seed=0):
    """Pick a run with `label` from any file, preferring unprompted runs, then runs with no other labels."""
    df = runs[runs["labels"].map(lambda ls: label in ls)]
    if (df.run_source == "unprompted").any():
        df = df[df.run_source == "unprompted"]
    if (df["labels"].map(len) == 1).any():
        df = df[df["labels"].map(len) == 1]
    return int(df.sample(1, random_state=seed).run_id.item())


def _md_message(msg, max_chars):
    """One message as a bold role tag plus a fenced block, reusing _fmt's truncation and tool-call formatting."""
    tag, body = _fmt(msg, max_chars).split("\n", 1)
    fence = "`" * max(3, max((len(m) for m in re.findall(r"`+", body)), default=0) + 1)
    return f"**{tag}**\n\n{fence}text\n{body.rstrip()}\n{fence}\n"


def sample_markdown(run, label, sample=-1, max_chars=1500):
    """Markdown equivalent of show_sample."""
    m, s = run["metadata"], run["samples"][sample]
    lines = [
        f"## {label}",
        "",
        f"run {m['run_id']} · {m['model']} · `{m['task_id']}` · {m['run_source']} · labels: {', '.join(m['labels'])}",
        "",
        f"*sample {sample} of {len(run['samples'])}: {len(s['input'])} input messages, {len(s['output'])} completion(s)*",
        "",
        "### Input",
        "",
    ]
    lines += [_md_message(msg, max_chars) for msg in s["input"]]
    for i, completion in enumerate(s["output"]):
        lines += [f"### Output {i}", ""] + [_md_message(msg, max_chars) for msg in completion]
    return "\n".join(lines)


examples = pd.DataFrame({"label": EXAMPLE_LABELS, "run_id": [pick_example(l, seed=0) for l in EXAMPLE_LABELS]})
examples = examples.merge(runs[["run_id", "model", "task_id", "run_source", "labels"]], on="run_id")

summary = ["| label | run | model | task | source |", "|---|---|---|---|---|"] + [
    f"| [{r.label}](#{r.label}) | {r.run_id} | {r.model} | `{r.task_id}` | {r.run_source} |"
    for r in examples.itertuples()
]
EXAMPLES_PATH.parent.mkdir(exist_ok=True)
EXAMPLES_PATH.write_text(
    "# MALT example transcripts\n\n" + "\n".join(summary) + "\n\n"
    + "\n\n".join(sample_markdown(load_run(r.run_id), r.label) for r in examples.itertuples())
)
print(f"Wrote {EXAMPLES_PATH.resolve()} ({EXAMPLES_PATH.stat().st_size / 1e6:.1f} MB)")
examples